In [ ]:
import os
import re
import warnings
import logging
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from transformers.integrations import TensorBoardCallback
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, confusion_matrix

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

# Set to "EKMAN_7" for the 7 consolidated classes, or "GOEMOTIONS_28" for the full 28 classes
CLASSIFICATION_MODE = "EKMAN_7"

# Change your file path here
train_path = "path/to/your_train_dataset.tsv"
test_path  = "path/to/your_test_dataset.tsv"



In [ ]:
ekman_labels = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']

goemotions_labels = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

if CLASSIFICATION_MODE == "EKMAN_7":
    emotion_labels = ekman_labels
else:
    emotion_labels = goemotions_labels

num_labels = len(emotion_labels)

def load_tsv_safe(filepath):
    rows = []
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.rstrip('\r\n')
            if not line.strip():
                continue
            parts = line.rsplit('\t', 2)
            if len(parts) == 3:
                rows.append(parts)
            elif len(parts) == 2:
                rows.append(["", parts[0], parts[1]])
    return pd.DataFrame(rows, columns=['text', 'label_ids', 'comment_id'])

try:
    df_train = load_tsv_safe(train_path)
    df_test = load_tsv_safe(test_path)
    print(f"Loaded datasets. Active classification mode: {CLASSIFICATION_MODE} ({num_labels} classes).")
except FileNotFoundError:
    print(f"ERROR: Files not found at: {base_path}")
    raise


In [ ]:
orig_28_to_ekman_7 = {
    2: 0, 3: 0, 10: 0,                                                       # 0: anger
    11: 1,                                                                   # 1: disgust
    14: 2, 19: 2,                                                            # 2: fear
    0: 3, 1: 3, 4: 3, 5: 3, 8: 3, 13: 3, 15: 3, 17: 3, 18: 3, 20: 3, 21: 3, 23: 3,  # 3: joy
    9: 4, 12: 4, 16: 4, 24: 4, 25: 4,                                        # 4: sadness
    6: 5, 7: 5, 22: 5, 26: 5,                                                # 5: surprise
    27: 6                                                                    # 6: neutral
}

def process_labels(df):
    try:
        int_labels = [list(map(int, str(label).split(','))) for label in df['label_ids']]
        binary_labels = np.zeros((len(df), num_labels))

        for i, labels in enumerate(int_labels):
            for label_idx in labels:
                if CLASSIFICATION_MODE == "EKMAN_7":
                    if label_idx in orig_28_to_ekman_7:
                        ekman_idx = orig_28_to_ekman_7[label_idx]
                        binary_labels[i, ekman_idx] = 1.0
                else:
                    if 0 <= label_idx < num_labels:
                        binary_labels[i, label_idx] = 1.0

        df['labels'] = list(binary_labels)
    except Exception as e:
        print(f"Error processing labels: {e}")
        raise
    return df

df_train = process_labels(df_train)
df_test = process_labels(df_test)

train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)


In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_data(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

train_dataset = train_dataset.remove_columns(['text', 'label_ids', 'comment_id'])
test_dataset = test_dataset.remove_columns(['text', 'label_ids', 'comment_id'])

train_dataset.set_format("torch")
test_dataset.set_format("torch")
print("Datasets tokenized and formatted for PyTorch.")


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

def multi_label_metrics(predictions):
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions.predictions))

    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= 0.5)] = 1
    y_true = predictions.label_ids

    f1_micro = f1_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    balanced_accuracies = []
    for i in range(y_true.shape[1]):
        bal_acc = balanced_accuracy_score(y_true[:, i], y_pred[:, i])
        balanced_accuracies.append(bal_acc)

    metrics = {
        'f1_micro': f1_micro,
        'accuracy': accuracy,
        'balanced_accuracy': np.mean(balanced_accuracies)
    }
    return metrics


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=300,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    disable_tqdm=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=multi_label_metrics,
)

print(f"Starting training for {CLASSIFICATION_MODE} mode...")
trainer.train()
print("Training finished.")


In [ ]:
print("Generating Confusion Matrix")
pred_output = trainer.predict(test_dataset)

sigmoid = torch.nn.Sigmoid()
probs = sigmoid(torch.Tensor(pred_output.predictions)).numpy()
y_true_matrix = pred_output.label_ids

y_true_single = np.argmax(y_true_matrix, axis=1)
y_pred_single = np.argmax(probs, axis=1)

cm = confusion_matrix(y_true_single, y_pred_single, labels=range(num_labels))

figsize = (18, 15) if CLASSIFICATION_MODE == "GOEMOTIONS_28" else (8, 6)
annot_kws = {"size": 7} if CLASSIFICATION_MODE == "GOEMOTIONS_28" else None

plt.figure(figsize=figsize)
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=emotion_labels,
    yticklabels=emotion_labels,
    annot_kws=annot_kws
)
plt.title(f'{CLASSIFICATION_MODE} Emotion Confusion Matrix')
plt.xlabel('Predicted Emotion')
plt.ylabel('True Emotion')
if CLASSIFICATION_MODE == "GOEMOTIONS_28":
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
